In [1]:
"""
Preprocess and scale CICIDS2017 (source domain). Save scaler for reuse on target
dataset, and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [2]:
### Import and concatenate CSVs ###

# Creates a Path object pointing to the source-domain CSV directory.
data_dir = Path("data/raw/source")

# Read each CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Load all source CSV files into a list of DataFrames.
dfs = [read_csv_with_fallback(f) for f in data_dir.glob("*.csv")]

# Concatenate all DataFrames into one source-domain DataFrame.
df = pd.concat(dfs, ignore_index=True)

# Display a quick shape check and preview rows.
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [3]:
### Data sanitization ###

# Handle missing values by replacing all occurrences of infinity with NaN
# then removing rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# remove duplicate flows and irrelevant columns
df.drop_duplicates(inplace=True)
df = df.drop(columns=["Flow ID", "Source IP", "Destination IP", "Timestamp"], errors="ignore")

# Remove leading/trailing spaces from all column names
df.rename(columns=lambda x: x.strip(), inplace=True)

In [4]:
### Feature-space alignment ###
# (select features according to predetermined shared feature space)

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")

# Handle missing file
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features. Order must be preserved.
shared_features = load_feature_order(FEATURE_LIST_PATH)

# Identify label column and handle leading or trailing whitespace
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Aligns a DataFrame to the canonical shared feature contract by dropping extra
# columns, checking/optionally filling missing columns, and enforcing exact order.
# Used when source and target datasets must produce identical model input schema.
def align_feature_space(frame, feature_list, fill_missing=False, fill_value=0.0):
    
    # Compare incoming columns against the expected shared feature contract.
    feature_list = list(feature_list)
    incoming = set(frame.columns)
    expected = set(feature_list)

    extra = sorted(incoming - expected)
    missing = sorted(expected - incoming)

    # Strict mode (default): block pipeline if required features are absent.
    # This protects training/inference consistency across datasets.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Missing required features: {preview} (total={len(missing)})"
        )

    # Optional tolerant mode: create absent columns with a fixed value, then
    # continue with the canonical ordering.
    if missing and fill_missing:
        for col in missing:
            frame[col] = fill_value

    # Drop extras and enforce exact column order expected by downstream steps.
    aligned = frame[feature_list].copy()
    return aligned, extra, missing

# Ensure index is contiguous before splitting/rejoining features and labels.
df = df.reset_index(drop=True)

# Align only feature columns; label handling happens separately.
feature_df = df.drop(columns=[label_col]).copy()
aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    fill_missing=False,
    fill_value=0.0,
 )

# Reattach labels by position (not index label) to avoid accidental NaNs.
labels_aligned = df[[label_col]].reset_index(drop=True)
aligned_X = aligned_X.reset_index(drop=True)
if len(aligned_X) != len(labels_aligned):
    raise ValueError(
        f"Feature/label row count mismatch after alignment: "
        f"X={len(aligned_X)}, y={len(labels_aligned)}"
    )
df = pd.concat([aligned_X, labels_aligned], axis=1)

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Aligned feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")
print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")

Loaded shared feature list from data/processed/shared_feature_space.json
Aligned feature count: 68
Dropped extra columns: 10
Missing required columns: 0
Missing labels after reattach: 0


In [5]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
SOURCE_LABEL_MAP_PATH = Path("data/processed/source_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not SOURCE_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Source label map file not found at {SOURCE_LABEL_MAP_PATH}"
    )

with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_space = json.load(f)
with open(SOURCE_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    source_label_map = json.load(f)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_target_classes = sorted(
    set(source_label_map.values()) - set(shared_label_space)
)
if invalid_target_classes:
    raise ValueError(
        "source_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_target_classes}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(source_label_map)

# Fail fast on unmapped non-missing raw labels to avoid silent label drift.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
if unmapped_mask.any():
    unmapped_indices = df.index[unmapped_mask].tolist()
    unmapped_raw = raw_labels[unmapped_mask].tolist()
    unique_unmapped_raw = sorted(set(unmapped_raw))
    preview_pairs = list(zip(unmapped_indices, [repr(v) for v in unmapped_raw]))[:10]

    raise ValueError(
        f"Unmapped raw labels found: {[repr(v) for v in unique_unmapped_raw[:10]]} "
        f"(total unique={len(unique_unmapped_raw)}, total rows={len(unmapped_indices)}). "
        f"Sample row/value pairs: {preview_pairs}. "
        f"Update {SOURCE_LABEL_MAP_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Sanity check: print class-frequency table after label alignment.
label_counts = df[label_col].value_counts(dropna=False).sort_values(ascending=False)
label_freq = (label_counts / len(df) * 100).round(2)

print("Label category frequencies after alignment:")
for cls in label_counts.index:
    print(f"- {cls}: {int(label_counts[cls])} ({label_freq[cls]:.2f}%)")

Label category frequencies after alignment:
- Benign: 2095057 (83.11%)
- Denial of Service: 321759 (12.76%)
- Probing: 90694 (3.60%)
- Brute Force: 10620 (0.42%)
- Malware: 1995 (0.08%)
- Web Injection: 673 (0.03%)


In [6]:
### Train/Test Split ###

from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Create feature matrix X and target vector y.
X = df.drop(columns=[label_col])
y = df[label_col]

# Single split: 80% train, 20% test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
 )

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 2016638/504160


In [7]:
### Scaling (save scaler for CIC_ToN_IoT) ###
# Note: applies scaler to data; exported data is already scaled

# Fit scaler on training features only to avoid data leakage, then apply to test split.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve feature names/indexing.
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Save fitted scaler for reuse in target-domain preprocessing/inference.
scaler_path = Path("models/source_scaler.joblib")
scaler_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_path)

print(f"Saved scaler to {scaler_path}")
print(f"Scaled splits shapes: train={X_train.shape}, test={X_test.shape}")

Saved scaler to models/source_scaler.joblib
Scaled splits shapes: train=(2016638, 68), test=(504160, 68)


In [8]:
### Label encoding (save encoder for CIC_ToN_IoT) ###

# Fit label encoder on training labels and apply to test labels.
le = LabelEncoder()
y_train = pd.Series(le.fit_transform(y_train), index=y_train.index, name=label_col)
y_test = pd.Series(le.transform(y_test), index=y_test.index, name=label_col)

# Save fitted label encoder for reuse in target-domain preprocessing/inference.
encoder_path = Path("models/label_encoder.joblib")
encoder_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(le, encoder_path)

print(f"Saved label encoder to {encoder_path}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Saved label encoder to models/label_encoder.joblib
Encoded classes (6): ['Benign', 'Brute Force', 'Denial of Service', 'Malware', 'Probing', 'Web Injection']


In [9]:
### Calculate and export covariance and mean statistics ###
# Note: calculated from training split only.

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Ensure exact feature order before CORAL statistics.
X_train_aligned = X_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_src = X_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
source_feature_mean = np.mean(X_src, axis=0)

# 2) Centered source data.
X_src_centered = X_src - source_feature_mean

# 3) Covariance matrix (core CORAL statistic).
source_covariance = np.cov(X_src_centered, rowvar=False, dtype=np.float64)
source_covariance = (source_covariance + source_covariance.T) / 2.0

# Regularize covariance before export so downstream CORAL stays numerically stable.
source_eigenvalues = np.linalg.eigvalsh(source_covariance)
source_min_eig = float(source_eigenvalues.min())
source_covariance_ridge = max(1e-6, float(-source_min_eig + 1e-6)) if source_min_eig <= 0 else 1e-6
source_covariance = source_covariance + np.eye(len(feature_order), dtype=np.float64) * source_covariance_ridge

# Sanity checks.
assert source_covariance.shape[0] == source_covariance.shape[1], "Covariance matrix must be square"
assert source_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_source_stats = {
    "feature_order": feature_order,
    "mean": source_feature_mean,
    "covariance": source_covariance,
    "min_eigenvalue_before_regularization": source_min_eig,
    "covariance_ridge": source_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_source_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_source_stats, coral_stats_path)

print("CORAL source statistics extracted and saved successfully.")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {source_covariance.shape}")
print(f"Min eigenvalue before regularization: {source_min_eig:.6e}")
print(f"Ridge added to covariance diagonal: {source_covariance_ridge:.6e}")

CORAL source statistics extracted and saved successfully.
Saved to: models/coral_source_stats.joblib
Features: 68
Covariance shape: (68, 68)
Min eigenvalue before regularization: -5.367154e-16
Ridge added to covariance diagonal: 1.000000e-06


In [10]:
### Export processed data ###

# Create output directory for processed source splits.
output_dir = Path("data/processed/source")
output_dir.mkdir(parents=True, exist_ok=True)

# Save training data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "train.csv", index=False)

# Save test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(y_train)} samples")
print(f"  Test: {len(y_test)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 2016638 samples
  Test: 504160 samples
  Output directory: data/processed/source
